# Maps Lead Generator — Google Colab Setup

This notebook runs the entire Crewai-maps-Scrapper codebase in Google Colab.

**Setup steps:**
1. Install dependencies
2. Set up Playwright with browser
3. Upload your codebase
4. Run the scraper

## Step 1: Install Dependencies

In [16]:
!pip install playwright playwright-stealth geopy pydantic pydantic-settings typer rich aiohttp python-dotenv nest_asyncio -q
print("✓ Dependencies installed")

✓ Dependencies installed


## Step 2: Install Playwright Browsers

In [18]:
import subprocess
import sys

# Install Chromium browser and its system dependencies
print("Installing Playwright Chromium and system dependencies...")
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
subprocess.run([sys.executable, "-m", "playwright", "install-deps"], check=True)

print("✓ Playwright Chromium and dependencies installed")

Installing Playwright Chromium and system dependencies...
✓ Playwright Chromium and dependencies installed


## Step 3: Upload Your Project Files

In [3]:
import os
from pathlib import Path

# Create working directory
os.makedirs("/content/scraper", exist_ok=True)
os.makedirs("/content/scraper/csv", exist_ok=True)
os.makedirs("/content/scraper/logs", exist_ok=True)

os.chdir("/content/scraper")
print(f"✓ Working directory: {os.getcwd()}")

✓ Working directory: /content/scraper


### Upload ZIP file with your code

In [6]:
from google.colab import files
import zipfile

print("Click 'Choose Files' to upload your crewai-scraper-colab.zip")
uploaded = files.upload()

# Extract ZIP if uploaded
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall("/content/scraper")
        print(f"✓ Extracted {filename}")

# If files were uploaded individually, they should be in current dir
print(f"✓ Files ready")

Click 'Choose Files' to upload your crewai-scraper-colab.zip


Saving crewai-scraper-colab.zip to crewai-scraper-colab (1).zip
✓ Extracted crewai-scraper-colab (1).zip
✓ Files ready


## Step 4: Verify Project Structure

In [7]:
import os
from pathlib import Path

os.chdir("/content/scraper")

# Check required files
required_files = [
    "src/main.py",
    "src/config.py",
    "src/tools/location_api.py",
    "src/tools/playwright_bot.py",
    "src/orchestration/pipeline.py",
    "requirements.txt"
]

print("📋 Checking project structure...\n")
all_good = True

for file_path in required_files:
    if Path(file_path).exists():
        print(f"✓ {file_path}")
    else:
        print(f"✗ {file_path} - MISSING")
        all_good = False

if all_good:
    print("\n✓ All files present! Ready to scrape.")
else:
    print("\n❌ Some files are missing. Check your upload.")

📋 Checking project structure...

✓ src/main.py
✓ src/config.py
✓ src/tools/location_api.py
✓ src/tools/playwright_bot.py
✓ src/orchestration/pipeline.py
✓ requirements.txt

✓ All files present! Ready to scrape.


## Step 5: Configure Search Parameters

In [10]:
# Set your search parameters here
TARGET_AREA = "Houston"  # City or country name
BUSINESS_TYPE = "dentist"  # What you're looking for
MAX_DISPLAY = 30  # Number of results to show in table
HEADLESS_MODE = True  # Keep True for Colab

print(f"Configuration:")
print(f"  Area: {TARGET_AREA}")
print(f"  Business: {BUSINESS_TYPE}")
print(f"  Max Display: {MAX_DISPLAY}")
print(f"  Headless: {HEADLESS_MODE}")

Configuration:
  Area: Huston
  Business: dentist
  Max Display: 30
  Headless: True


## Step 6: Discover Locations

In [11]:
import sys
sys.path.insert(0, '/content/scraper')

from src.tools.location_api import LocationDiscoveryTool

print(f"🔍 Discovering locations in {TARGET_AREA}...\n")

location_tool = LocationDiscoveryTool()
locations = location_tool.discover_locations(TARGET_AREA)

if not locations:
    print(f"❌ No locations found for '{TARGET_AREA}'. Try a different query.")
else:
    print(f"✓ Found {len(locations)} location(s)\n")

    for idx, loc in enumerate(locations, 1):
        print(f"{idx}. {loc.display_name}")
        print(f"   Type: {loc.place_type}\n")

🔍 Discovering locations in Huston...

✓ Found 11 location(s)

1. Huston, Missoula County, Montana, United States
   Type: peak

2. Huston, Lower Turkeyfoot Township, Somerset County, Pennsylvania, 15424, United States
   Type: hamlet

3. Huston, Canyon County, Idaho, United States
   Type: hamlet

4. Huston, Yates Township, Lake County, Michigan, 49642, United States
   Type: residential

5. Huston, Yates Township, Lake County, Michigan, 49304, United States
   Type: residential

6. Huston, Waldron, Scott County, Arkansas, 72958, United States
   Type: residential

7. Huston, Cloverport, Breckinridge County, Kentucky, 40111, United States
   Type: residential

8. Huston Township, Clearfield County, Pennsylvania, 15849, United States
   Type: administrative

9. Huston Township, Centre County, Pennsylvania, 16844, United States
   Type: administrative

10. Huston Township, Blair County, Pennsylvania, United States
   Type: administrative

11. Houston, Harris County, Texas, United States


## Step 7: Select a Location

In [12]:
# Change this number to select a different location from the list above
LOCATION_NUMBER = 11

if LOCATION_NUMBER < 1 or LOCATION_NUMBER > len(locations):
    print(f"❌ Invalid selection. Choose between 1 and {len(locations)}.")
else:
    selected_location = locations[LOCATION_NUMBER - 1]
    location_name = selected_location.display_name.split(",")[0].strip()
    full_location = selected_location.display_name

    print(f"✓ Selected: {full_location}")
    print(f"  Type: {selected_location.place_type}")

✓ Selected: Houston, Harris County, Texas, United States
  Type: administrative


## Step 8: Run the Scraper

In [19]:
import sys
import os
import asyncio
import nest_asyncio

# Allow nested event loops in Colab
nest_asyncio.apply()

sys.path.insert(0, '/content/scraper')
os.chdir('/content/scraper')

from src.orchestration.pipeline import ScrapingPipeline

output_file = f"/content/scraper/csv/leads_{location_name.lower().replace(' ', '_')}_{BUSINESS_TYPE.lower().replace(' ', '_')}.csv"

print(f"""\n🔍 SCRAPER CONFIGURATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Location:    {full_location}
Business:    {BUSINESS_TYPE}
Output:      {output_file}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n""")

print("Starting scraper... (this may take 5-15 minutes)\n")

try:
    pipeline = ScrapingPipeline()

    # We call the internal async method directly to avoid the asyncio.run() conflict in Colab
    # Based on the error log, the method is likely _run_async or run_async
    if hasattr(pipeline, '_run_async'):
        results = asyncio.get_event_loop().run_until_complete(pipeline._run_async(
            location=full_location,
            business_type=BUSINESS_TYPE,
            output_file=output_file,
            place_type=selected_location.place_type
        ))
    else:
        # Fallback to a standard async execution of the main run if it was a coroutine
        results = asyncio.get_event_loop().run_until_complete(pipeline.run(
            location=full_location,
            business_type=BUSINESS_TYPE,
            output_file=output_file,
            place_type=selected_location.place_type
        ))

    if results:
        print(f"\n✓ Scraping complete! Found {len(results)} results.")
    else:
        print("\n⚠️ No results found. Google may be blocking or no listings exist.")

except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()


🔍 SCRAPER CONFIGURATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Location:    Houston, Harris County, Texas, United States
Business:    dentist
Output:      /content/scraper/csv/leads_houston_dentist.csv
━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Starting scraper... (this may take 5-15 minutes)


  BROAD AREA DETECTED — will scrape 20 cities one by one.


  ── [1/20] Houston, Houston, Harris County, Texas, United States ──

  OPENING: https://www.google.com/maps/search/dentist+in+Houston,+Houston,+Harris+County,+Texas,+United+States
  NAVIGATING to Google Maps...
  Scrolling (no result cap — stops at end of list)...
  STOPPED (no new results): 54 businesses.     

  FOUND 54 businesses. Extracting with 8 Playwright workers + 20 HTTP workers...

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 #     Business                   Rev    Phone                Email                        Website                  Address     

## Step 9: View Results

In [ ]:
import pandas as pd
from pathlib import Path

if Path(output_file).exists():
    df = pd.read_csv(output_file)
    print(f"✓ Total records: {len(df)}\n")

    # Display first MAX_DISPLAY rows
    display_df = df.head(MAX_DISPLAY)

    print(f"📊 Results (first {min(MAX_DISPLAY, len(df))} of {len(df)} records)\n")
    display(display_df)

    if len(df) > MAX_DISPLAY:
        print(f"\n... and {len(df) - MAX_DISPLAY} more records in your CSV")
else:
    print("❌ No CSV file found. Scraper may have failed.")

## Step 10: Download Results

In [ ]:
from google.colab import files
from pathlib import Path

if Path(output_file).exists():
    files.download(output_file)
    print(f"✓ Downloading: {Path(output_file).name}")
else:
    print("❌ No CSV file to download")

## Troubleshooting

**Module not found?** Make sure the ZIP was extracted properly.  
**Browser won't start?** Re-run Step 2.  
**No results?** Try a different location or wait 15 minutes before retrying.  
**CAPTCHA?** Google blocked you. Wait 30 minutes and try again.